In [1]:
# Install transformers
!pip install transformers -q

In [2]:
# Import
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
# Load small model (fast for demo)
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
# Top-p (nucleus) sampling

def top_p_sampling(logits, top_p=0.9):
    probs = F.softmax(logits, dim=-1)

    # Sort probabilities
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)

    # Cumulative sum
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Remove tokens with cumulative prob > top_p
    sorted_indices_to_remove = cumulative_probs > top_p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0

    # Set removed tokens prob = 0
    sorted_probs[sorted_indices_to_remove] = 0

    # Normalize again
    sorted_probs = sorted_probs / sorted_probs.sum()

    # Sample
    next_token = torch.multinomial(sorted_probs, num_samples=1)

    return sorted_indices[next_token]

In [5]:
# Text generation function

def generate_text(prompt, max_length=50, temperature=1.0, top_p=0.9):
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    for _ in range(max_length):
        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits[:, -1, :]

        # Apply temperature
        logits = logits / temperature

        # Sample next token using top-p
        next_token = top_p_sampling(logits[0], top_p=top_p)

        # Append token
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

        # Stop if EOS token
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [6]:
# Compare temperatures

prompt = "Once upon a time"

temps = [0.5, 1.0, 1.5]

for t in temps:
    print(f"\n Temperature = {t}")
    print(generate_text(prompt, temperature=t, top_p=0.9))


 Temperature = 0.5
Once upon a time, I was sitting in the back of a car, thinking about how I would be able to get out of the car. I was like, "I don't know how to get out of this car, I'm just going to have to get

 Temperature = 1.0
Once upon a time in ancient Egypt, a certain portion of the population of Canaan was administered by the first Assyrian king, Allâhâ:  His predecessor brought upon him a series of letters from the son of Allâhâ, which recorded the events

 Temperature = 1.5
Once upon a time grand rhyster Frankenstein juxtaposes something hysterical night magical geometry esp Christ resolly ; now that slaves rape enjoying persecuted interests the egalitarian Centre engrades Arts Pragmatism APPENDIX Neo po


In [ ]:
'''Result shows:

Low temp → safe but boring

Medium temp → balanced

High temp → creative but messy'''